# Notebook 41 — L1 Pendulum: InfoNCE + Barlow Twins
**Runtime:** Colab T4 GPU. **Prerequisite:** notebook 39; run 40 for the NSV comparison.

`infonce`: cosine symmetric cross-view contrastive loss on temporal pairs.<br>
`bt_temporal`: redundancy reduction on shifted windows.<br>
`bt_same`: redundancy reduction on two observations of the same state window.

InfoNCE's positive target is always the matched row; overlapping nearby windows
from the same trajectory are masked as negatives. Distant recurrence can still
create false negatives, which is part of the scientific comparison.
Save backbone h and projector z separately; InfoNCE also saves normalized z_unit.


## Protocol shared by notebooks 40 and 41

All methods use the same CNN backbone, frame history, trajectory split, initial
encoder seed and evaluation rows. Default window = lag = 5: temporal views do not
overlap. A separate `view_noise_std=0.01` training augmentation is recorded even
for clean L1a; all input branches receive it. To study zero augmentation, set it
explicitly to zero (BT-same then has identical views in L1a).

Observation noise is deterministic per frame and view, shared across methods;
same-time views use independent synthetic observation realizations. NSV targets
are future **observations**, not oracle clean frames or physical labels.
Stage/model selection uses validation only; test labels are first used in 42.
A completed run is reused by exact config hash. Incomplete runs are not overwritten;
inspect them or change `run_tag`. Full rerun/resume of partial epochs is not implemented.


## Cell 1 — Drive and isolated paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, tempfile, json

PROJECT_DIR = Path('/content/drive/MyDrive/BSISO_SSL_Project')
L1_ROOT = PROJECT_DIR / 'toy' / 'pendulum'
L1_ROOT.mkdir(parents=True, exist_ok=True)
# Embedded code needs no GitHub clone/download or external .py upload.
RUNTIME_DIR = Path(tempfile.mkdtemp(prefix='ddcs_l1_code_'))
os.chdir(RUNTIME_DIR)
sys.path.insert(0, str(RUNTIME_DIR))
for name in ('l1_data', 'l1_learning', 'l1_evaluation'):
    sys.modules.pop(name, None)
CACHE_DIR = Path('/content/ddcs_l1_frames')  # disposable; never store frame arrays on Drive

def resolve_dataset_id(value):
    if value:
        if not (L1_ROOT / 'datasets' / value / 'manifest.json').exists():
            raise FileNotFoundError(f'Dataset {value} not found; run notebook 39 first.')
        return value
    available = sorted(p.parent.name for p in (L1_ROOT / 'datasets').glob('*/manifest.json'))
    if len(available) != 1:
        raise ValueError(f'Set DATASET_ID explicitly from notebook 39. Available: {available}')
    return available[0]

print('Drive outputs:', L1_ROOT)
print('Local code:', RUNTIME_DIR)


## Cell 2 — Embedded implementations


In [ ]:
%%writefile l1_data.py
"""L1 pendulum simulation, observation, split and provenance utilities (no torch)."""
from __future__ import annotations

from dataclasses import asdict, dataclass
import hashlib
import json
from pathlib import Path

import numpy as np

SCHEMA = "pendulum-l1-v1"
SPLITS = {"train": 0, "val": 1, "test": 2}


def json_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
    tmp.replace(path)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:16]


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


@dataclass(frozen=True)
class SimulationConfig:
    seed: int = 20261027
    n_fixed: int = 100
    n_length_groups: int = 60
    trajectories_per_length: int = 5
    periods: float = 8.0
    frames_per_period: int = 50
    substeps: int = 8
    g: float = 1.0
    reference_length: float = 1.0
    damping: float = 0.02
    length_min: float = 0.8
    length_max: float = 1.2
    image_size: int = 64

    def validate(self):
        if self.n_fixed < 5 or self.n_length_groups < 5:
            raise ValueError("Need at least five independent groups for train/val/test.")
        if self.trajectories_per_length < 1 or self.periods <= 0:
            raise ValueError("Invalid trajectory count/duration.")
        if self.frames_per_period < 10 or self.substeps < 1:
            raise ValueError("Insufficient sampling/integration resolution.")
        if self.g <= 0 or self.reference_length <= 0 or self.damping < 0:
            raise ValueError("Invalid physical parameters.")
        if not 0 < self.length_min < self.length_max <= 1.5 * self.reference_length:
            raise ValueError("Length interval must be positive and fit the fixed camera.")
        if self.image_size != 64:
            raise ValueError("The initial CNN/renderer protocol uses 64 x 64 images.")

    @property
    def period(self):
        return 2 * np.pi * np.sqrt(self.reference_length / self.g)

    @property
    def dt(self):
        return self.period / self.frames_per_period


def group_split(groups, seed):
    groups = np.asarray(groups)
    unique = np.unique(groups)
    if len(unique) < 5:
        raise ValueError("Too few independent groups.")
    order = np.random.default_rng(seed).permutation(unique)
    n_val = max(1, int(round(0.2 * len(order))))
    n_test = n_val
    mapping = {int(g): 0 for g in order}
    mapping.update({int(g): 1 for g in order[-n_val-n_test:-n_test]})
    mapping.update({int(g): 2 for g in order[-n_test:]})
    return np.array([mapping[int(g)] for g in groups], dtype=np.int8)


def simulate(initial, lengths, times, g=1.0, damping=0.02, substeps=8):
    """Vectorized RK4, constant length per trajectory; no stochastic force in L1a-c."""
    state = np.asarray(initial, dtype=np.float64).copy()
    lengths = np.asarray(lengths, dtype=np.float64)
    times = np.asarray(times)
    if state.shape != (len(lengths), 2) or np.any(lengths <= 0):
        raise ValueError("Expected initial=(trajectories,2), positive lengths.")
    if substeps < 1 or len(times) < 2 or np.any(np.diff(times) <= 0):
        raise ValueError("Invalid time grid or substeps.")
    out = np.empty((len(lengths), len(times), 2), dtype=np.float64)
    out[:, 0] = state

    def rhs(y):
        return np.stack((y[:, 1], -damping*y[:, 1] - g/lengths*np.sin(y[:, 0])), axis=1)

    for i, interval in enumerate(np.diff(times), start=1):
        h = interval / substeps
        for _ in range(substeps):
            k1 = rhs(state)
            k2 = rhs(state + h*k1/2)
            k3 = rhs(state + h*k2/2)
            k4 = rhs(state + h*k3)
            state += h*(k1 + 2*k2 + 2*k3 + k4)/6
        out[:, i] = state
    return out


def energy(states, lengths, g=1.0):
    """Mechanical energy for unit mass (constant L), not a harmonic approximation."""
    length = np.asarray(lengths)[:, None]
    return 0.5*length**2*states[..., 1]**2 + g*length*(1-np.cos(states[..., 0]))


def make_bundle(cfg, variable_length=False):
    cfg.validate()
    rng = np.random.default_rng(cfg.seed + (1000 if variable_length else 0))
    if variable_length:
        group_lengths = rng.uniform(cfg.length_min, cfg.length_max, cfg.n_length_groups)
        groups = np.repeat(np.arange(cfg.n_length_groups), cfg.trajectories_per_length)
        lengths = group_lengths[groups]
    else:
        groups = np.arange(cfg.n_fixed)
        lengths = np.full(cfg.n_fixed, cfg.reference_length)
    initial = rng.uniform([-1.0, -0.5], [1.0, 0.5], size=(len(groups), 2))
    # Avoid a dataset dominated by almost stationary initial conditions.
    weak = np.linalg.norm(initial, axis=1) < 0.15
    while weak.any():
        initial[weak] = rng.uniform([-1.0, -0.5], [1.0, 0.5], size=(int(weak.sum()), 2))
        weak = np.linalg.norm(initial, axis=1) < 0.15
    times = np.arange(int(round(cfg.periods*cfg.frames_per_period)) + 1)*cfg.dt
    return {
        "states": simulate(initial, lengths, times, cfg.g, cfg.damping, cfg.substeps),
        "times": times, "lengths": lengths, "groups": groups.astype(np.int32),
        "splits": group_split(groups, cfg.seed + 17),
    }


def numerical_checks(bundle, cfg):
    original = bundle["states"][:3]
    fine = simulate(original[:, 0], bundle["lengths"][:3], bundle["times"],
                    cfg.g, cfg.damping, cfg.substeps*2)
    err = float(np.max(np.abs(original-fine)))
    e = energy(bundle["states"], bundle["lengths"], cfg.g)
    max_rise = float(np.max(np.diff(e, axis=1)))
    if err > 1e-5 or max_rise > 1e-8:
        raise ValueError(f"Integrator check failed: convergence={err}, energy rise={max_rise}")
    for group in np.unique(bundle["groups"]):
        if len(np.unique(bundle["splits"][bundle["groups"] == group])) != 1:
            raise ValueError("A group crossed splits.")
    return {"rk4_step_halving_max_error": err, "max_energy_increment": max_rise,
            "n_trajectories": len(e), "frames_per_trajectory": e.shape[1]}


def create_dataset(root, cfg):
    """Small truth arrays on Drive; frames are rendered only into disposable Colab cache."""
    cfg.validate()
    identity = {"schema": SCHEMA, "simulation": asdict(cfg), "source_sha": file_sha(__file__)}
    dataset_id = digest(identity)
    out = Path(root)/"datasets"/dataset_id
    if (out/"manifest.json").exists():
        load_dataset(root, dataset_id, "fixed")  # Validate existing hashes, never overwrite.
        load_dataset(root, dataset_id, "variable")
        return dataset_id
    if out.exists():
        raise FileExistsError(f"Incomplete dataset exists: {out}; use a new seed or inspect it.")
    out.mkdir(parents=True)
    checks, hashes = {}, {}
    for name, variable in [("fixed", False), ("variable", True)]:
        bundle = make_bundle(cfg, variable)
        checks[name] = numerical_checks(bundle, cfg)
        np.savez_compressed(out/f"{name}.npz", **bundle)
        hashes[name] = file_sha(out/f"{name}.npz")
    json_write(out/"manifest.json", {**identity, "dataset_id": dataset_id,
                                    "checks": checks, "data_sha256": hashes})
    return dataset_id


def load_dataset(root, dataset_id, kind):
    if kind not in {"fixed", "variable"}:
        raise ValueError(kind)
    out = Path(root)/"datasets"/dataset_id
    manifest = json.loads((out/"manifest.json").read_text())
    if manifest["schema"] != SCHEMA or manifest["dataset_id"] != dataset_id:
        raise ValueError("Dataset identity/schema mismatch.")
    if manifest["source_sha"] != file_sha(__file__):
        raise ValueError("Data code differs from notebook 39; use the same notebook release.")
    if file_sha(out/f"{kind}.npz") != manifest["data_sha256"][kind]:
        raise ValueError("Dataset file checksum mismatch.")
    with np.load(out/f"{kind}.npz", allow_pickle=False) as data:
        bundle = {k: data[k] for k in data.files}
    return bundle, SimulationConfig(**manifest["simulation"]), manifest


CONDITIONS = {
    "L1a": {"kind": "fixed", "noise_std": 0.0, "visible_length": True},
    "L1b_s002": {"kind": "fixed", "noise_std": 0.02, "visible_length": True},
    "L1b_s005": {"kind": "fixed", "noise_std": 0.05, "visible_length": True},
    "L1c_visible": {"kind": "variable", "noise_std": 0.02, "visible_length": True},
    "L1c_hidden": {"kind": "variable", "noise_std": 0.02, "visible_length": False},
}


def render(theta, lengths, size=64, visible_length=True, reference_length=1.0):
    """Fixed camera, Gaussian anti-aliased rod and bob; returns (N,H,W) in [0,1]."""
    theta = np.atleast_1d(theta)
    lengths = np.broadcast_to(lengths, theta.shape)
    radius = 0.42*(size-1)*(lengths/reference_length if visible_length else np.ones_like(theta))
    px, py = (size-1)/2, 0.14*(size-1)
    dx, dy = radius*np.sin(theta), radius*np.cos(theta)
    yy, xx = np.mgrid[:size, :size]
    rx, ry = xx[None]-px, yy[None]-py
    a = np.clip((rx*dx[:, None, None]+ry*dy[:, None, None]) /
                radius[:, None, None]**2, 0, 1)
    d2 = (rx-a*dx[:, None, None])**2+(ry-a*dy[:, None, None])**2
    rod = 0.45*np.exp(-d2/(2*0.65**2))
    bob = np.exp(-((rx-dx[:, None, None])**2+(ry-dy[:, None, None])**2)/(2*1.65**2))
    pivot = 0.7*np.exp(-(rx**2+ry**2)/(2*1.1**2))
    return np.maximum(np.maximum(rod, bob), pivot).astype(np.float32)


def frame_cache(bundle, cfg, condition, cache_root, dataset_id):
    spec = CONDITIONS[condition]
    name = f"{dataset_id}_{spec['kind']}_{'visible' if spec['visible_length'] else 'hidden'}"
    path = Path(cache_root)/f"{name}.npy"
    shape = (*bundle["states"].shape[:2], cfg.image_size, cfg.image_size)
    if path.exists():
        arr = np.load(path, mmap_mode="r", allow_pickle=False)
        if arr.shape != shape or arr.dtype != np.uint8:
            raise ValueError("Frame cache shape/dtype mismatch; remove this disposable cache.")
        return arr
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".partial.npy")
    arr = np.lib.format.open_memmap(tmp, mode="w+", dtype="uint8", shape=shape)
    for tr in range(shape[0]):
        arr[tr] = np.rint(255*render(bundle["states"][tr, :, 0], bundle["lengths"][tr],
                                   cfg.image_size, spec["visible_length"], cfg.reference_length)).astype("uint8")
    arr.flush()
    del arr
    tmp.replace(path)
    return np.load(path, mmap_mode="r", allow_pickle=False)


def index_windows(bundle, window=5, future=5, stride=2):
    """Each row is (trajectory, final input frame); all windows remain in one trajectory."""
    if min(window, stride) < 1 or future < 0:
        raise ValueError("Invalid window/stride/future.")
    ends = np.arange(window-1, bundle["states"].shape[1]-future, stride)
    if not len(ends):
        raise ValueError("Trajectories are too short for the requested history/horizon.")
    n = len(bundle["states"])
    return np.column_stack((np.repeat(np.arange(n), len(ends)), np.tile(ends, n))).astype("int32")


def observation(frames, trajectory, end, window, noise_std, seed, draw=0):
    """Noise is keyed by frame: overlapping windows share the same observed pixels."""
    start = int(end)-window+1
    if start < 0 or end >= frames.shape[1]:
        raise IndexError("Window outside trajectory.")
    out = np.array(frames[int(trajectory), start:int(end)+1], dtype="float32")/255
    if noise_std:
        for j, t in enumerate(range(start, int(end)+1)):
            rng = np.random.default_rng(np.random.SeedSequence([seed, draw, int(trajectory), t]))
            out[j] += noise_std*rng.standard_normal(out[j].shape).astype("float32")
    # Do not clip or normalize individual frames: keep the stated zero-mean noise model.
    return out


def sample_truth(bundle, index, horizon=0):
    tr, t = index.T
    return np.column_stack((bundle["states"][tr, t+horizon], bundle["lengths"][tr]))


def estimate_id(features, groups, ks=(10, 20, 40), max_points=2000, seed=17):
    """Train-only kNN MLE diagnostic; exclude neighbors from the same trajectory/group.

    This is a finite-sample, scale-dependent estimate, not a guaranteed physical dimension.
    No per-coordinate whitening: it would amplify near-null latent directions.
    """
    from sklearn.neighbors import NearestNeighbors
    z = np.asarray(features, dtype="float64")
    groups = np.asarray(groups)
    if len(z) != len(groups) or z.ndim != 2 or not np.isfinite(z).all():
        raise ValueError("Invalid ID input.")
    take = np.random.default_rng(seed).choice(len(z), min(len(z), max_points), replace=False)
    z, groups = z[take], groups[take]
    candidates = min(len(z), max(128, max(ks)*8))
    distances, neighbors = NearestNeighbors(n_neighbors=candidates).fit(z).kneighbors(z)
    rows = []
    for k in ks:
        estimates = []
        for i in range(len(z)):
            valid = (groups[neighbors[i]] != groups[i]) & (distances[i] > 1e-10)
            d = distances[i][valid][:k]
            if len(d) == k:
                denominator = np.log(d[-1]/d[:-1]).sum()
                if denominator > 1e-10:
                    estimates.append((k-2)/denominator)
        rows.append({"k": int(k), "n_valid": len(estimates),
                     "median_id": float(np.median(estimates)) if estimates else None,
                     "mean_id": float(np.mean(estimates)) if estimates else None})
    singular = np.linalg.svd(z-z.mean(0), compute_uv=False)**2
    pr = float(singular.sum()**2/(np.square(singular).sum()+1e-30))
    return {"n_sampled": len(z), "scales": rows, "participation_ratio": pr,
            "neighbor_exclusion": "same group", "used_split": "train"}


In [ ]:
%%writefile l1_learning.py
"""Shared Colab training for predictive NSV, InfoNCE and two Barlow view protocols."""
from __future__ import annotations

from dataclasses import asdict, dataclass
import copy
import json
import math
from pathlib import Path
import platform
import random
import time

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset

from l1_data import (CONDITIONS, digest, estimate_id, file_sha, frame_cache, index_windows,
                     json_write, load_dataset, observation)


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 42
    window: int = 5
    lag: int = 5
    train_stride: int = 2
    eval_stride: int = 4
    hidden_dim: int = 64
    projection_dim: int = 2
    batch_size: int = 128
    epochs: int = 30
    refine_epochs: int = 60
    refine_dims: tuple = (1, 2, 3, 4, 8)
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    temperature: float = 0.1
    barlow_offdiag: float = 0.005
    view_noise_std: float = 0.01
    run_tag: str = "pilot"

    def validate(self):
        if min(self.window, self.lag, self.train_stride, self.eval_stride, self.hidden_dim,
               self.projection_dim, self.epochs, self.refine_epochs) < 1:
            raise ValueError("Counts/lags/dimensions must be positive.")
        if self.batch_size < 2 or self.temperature <= 0 or self.learning_rate <= 0:
            raise ValueError("Invalid batch size, temperature, or learning rate.")
        if self.view_noise_std < 0 or self.barlow_offdiag < 0:
            raise ValueError("Noise/penalties must be nonnegative.")
        if not self.refine_dims or min(self.refine_dims) < 1:
            raise ValueError("Specify positive NSV bottleneck dimensions.")


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class WindowDataset(Dataset):
    """Physical state labels are deliberately absent from all training batches."""
    def __init__(self, frames, index, condition, data_seed, config, mode="embedding", augment=False):
        self.frames, self.index = frames, np.asarray(index)
        self.spec, self.seed, self.cfg = CONDITIONS[condition], data_seed, config
        self.mode, self.augment, self.epoch = mode, augment, 0

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        tr, t = map(int, self.index[i])
        x = observation(self.frames, tr, t, self.cfg.window, self.spec["noise_std"], self.seed, 0)
        if self.mode == "embedding":
            return torch.from_numpy(x)
        shift = 0 if self.mode == "same" else self.cfg.lag
        y = observation(self.frames, tr, t+shift, self.cfg.window,
                        self.spec["noise_std"], self.seed, 1)
        if self.augment and self.cfg.view_noise_std:
            rng = np.random.default_rng(np.random.SeedSequence([self.cfg.seed, self.epoch, int(i), 301]))
            x += self.cfg.view_noise_std*rng.standard_normal(x.shape).astype("float32")
            # Prediction targets are future observations, never oracle clean images.
            if self.mode != "predict":
                y += self.cfg.view_noise_std*rng.standard_normal(y.shape).astype("float32")
        return torch.from_numpy(x), torch.from_numpy(y), tr, t


class Encoder(nn.Module):
    def __init__(self, window=5, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(window, 16, 5, 2, 2), nn.GroupNorm(4, 16), nn.SiLU(),
            nn.Conv2d(16, 32, 3, 2, 1), nn.GroupNorm(4, 32), nn.SiLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.GroupNorm(4, 64), nn.SiLU(),
            nn.Flatten(), nn.Linear(64*8*8, hidden_dim))

    def forward(self, x):
        return self.net(x)


class Decoder(nn.Module):
    def __init__(self, hidden_dim, window):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, 64*8*8)
        self.net = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, 3, padding=1), nn.SiLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(32, 16, 3, padding=1), nn.SiLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(16, window, 3, padding=1))

    def forward(self, h):
        # Linear output accommodates zero-mean observation noise outside [0,1].
        return self.net(self.fc(h).reshape(-1, 64, 8, 8))


class Representation(nn.Module):
    def __init__(self, cfg, predictive=False):
        super().__init__()
        self.encoder = Encoder(cfg.window, cfg.hidden_dim)
        self.head = Decoder(cfg.hidden_dim, cfg.window) if predictive else nn.Sequential(
            nn.Linear(cfg.hidden_dim, 128), nn.LayerNorm(128), nn.SiLU(),
            nn.Linear(128, cfg.projection_dim))

    def forward(self, x):
        h = self.encoder(x)
        return h, self.head(h)


def infonce_loss(a, b, trajectory, ends, exclusion, temperature=0.1):
    """Symmetric cross-view InfoNCE; the same row is the sole designated positive."""
    if len(a) < 2:
        raise ValueError("InfoNCE needs at least two pairs.")
    a, b = F.normalize(a, dim=1), F.normalize(b, dim=1)
    logits = a @ b.T / temperature
    near = (trajectory[:, None] == trajectory[None, :]) & ((ends[:, None]-ends[None, :]).abs() < exclusion)
    diagonal = torch.eye(len(a), dtype=torch.bool, device=a.device)
    logits = logits.masked_fill(near & ~diagonal, -torch.inf)
    target = torch.arange(len(a), device=a.device)
    return (F.cross_entropy(logits, target)+F.cross_entropy(logits.T, target))/2


def barlow_loss(a, b, offdiag=0.005):
    if len(a) < 2:
        raise ValueError("Cross-correlation needs at least two samples.")
    a = (a-a.mean(0))/torch.sqrt(a.var(0, unbiased=False)+1e-5)
    b = (b-b.mean(0))/torch.sqrt(b.var(0, unbiased=False)+1e-5)
    c = a.T @ b / len(a)
    diagonal = torch.diagonal(c)
    mask = ~torch.eye(c.shape[0], dtype=torch.bool, device=c.device)
    return (diagonal-1).square().sum()+offdiag*c[mask].square().sum()


def batch_loss(model, batch, method, cfg, device):
    x, y, tr, t = (v.to(device) for v in batch)
    _, za = model(x)
    if method == "nsv":
        return F.mse_loss(za, y)
    _, zb = model(y)
    if method == "infonce":
        return infonce_loss(za, zb, tr, t, cfg.window+cfg.lag, cfg.temperature)
    return barlow_loss(za, zb, cfg.barlow_offdiag)


def fit_representation(model, train_ds, val_ds, method, cfg, device, out):
    if min(len(train_ds), len(val_ds)) < 2:
        raise ValueError("Too few training/validation windows.")
    generator = torch.Generator().manual_seed(cfg.seed)
    train = DataLoader(train_ds, batch_size=min(cfg.batch_size, len(train_ds)), shuffle=True,
                       drop_last=True, num_workers=0, generator=generator)
    val = DataLoader(val_ds, batch_size=min(cfg.batch_size, len(val_ds)), shuffle=False,
                     drop_last=True, num_workers=0)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    best, best_state, history = math.inf, None, []
    start = time.time()
    for epoch in range(cfg.epochs):
        train_ds.epoch = epoch
        record = {"epoch": epoch+1}
        for name, loader in [("train", train), ("val", val)]:
            model.train(name == "train")
            total, count = 0.0, 0
            for batch in loader:
                with torch.set_grad_enabled(name == "train"):
                    loss = batch_loss(model, batch, method, cfg, device)
                    if not torch.isfinite(loss):
                        raise FloatingPointError(f"Nonfinite {method} loss at epoch {epoch+1}")
                    if name == "train":
                        optimizer.zero_grad(set_to_none=True)
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                        optimizer.step()
                total += float(loss.detach())*len(batch[0])
                count += len(batch[0])
            record[name] = total/count
        if record["val"] < best:
            best = record["val"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, Path(out)/"model_best.pt")
        record["elapsed_seconds"] = time.time()-start
        history.append(record)
        json_write(Path(out)/"training_history.json", history)
        print(f"{method} epoch {epoch+1:3d}: train={record['train']:.5f}, val={record['val']:.5f}", flush=True)
    model.load_state_dict(best_state)
    model.eval()
    return history


def encode(model, ds, device, batch_size=128, predictive=False):
    hs, zs = [], []
    model.eval()
    with torch.no_grad():
        for x in DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0):
            h = model.encoder(x.to(device))
            hs.append(h.cpu().numpy())
            if not predictive:
                zs.append(model.head(h).cpu().numpy())
    return np.concatenate(hs), None if predictive else np.concatenate(zs)


class SineLayer(nn.Module):
    def __init__(self, n_in, n_out, first=False):
        super().__init__()
        self.linear = nn.Linear(n_in, n_out)
        bound = 1/n_in if first else np.sqrt(6/n_in)/30
        with torch.no_grad():
            self.linear.weight.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(30*self.linear(x))


class Refiner(nn.Module):
    def __init__(self, h_train, dim):
        super().__init__()
        mean = h_train.mean(0)
        std = h_train.std(0).clip(min=max(float(h_train.std(0).max())*0.01, 1e-5))
        self.register_buffer("mean", torch.as_tensor(mean, dtype=torch.float32))
        self.register_buffer("scale", torch.as_tensor(std, dtype=torch.float32))
        n_in = h_train.shape[1]
        self.encoder = nn.Sequential(SineLayer(n_in, 128, first=True), SineLayer(128, 64), nn.Linear(64, dim))
        self.decoder = nn.Sequential(SineLayer(dim, 64, first=True), SineLayer(64, 128), nn.Linear(128, n_in))

    def forward(self, h):
        normalized = (h-self.mean)/self.scale
        z = self.encoder(normalized)
        return z, self.decoder(z), normalized


def refine(h, splits, dim, cfg, device, out):
    seed_all(cfg.seed+dim)
    model = Refiner(h[splits == 0], dim).to(device)
    loaders = {name: DataLoader(TensorDataset(torch.from_numpy(h[splits == split])),
                               batch_size=cfg.batch_size, shuffle=(split == 0), num_workers=0)
               for name, split in [("train", 0), ("val", 1)]}
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    history, best, best_state = [], math.inf, None
    for epoch in range(cfg.refine_epochs):
        row = {"epoch": epoch+1}
        for name, loader in loaders.items():
            model.train(name == "train")
            total, count = 0.0, 0
            for (x,) in loader:
                with torch.set_grad_enabled(name == "train"):
                    _, reconstructed, target = model(x.to(device))
                    loss = F.mse_loss(reconstructed, target)
                    if not torch.isfinite(loss):
                        raise FloatingPointError("Nonfinite NSV refine loss.")
                    if name == "train":
                        optimizer.zero_grad(set_to_none=True)
                        loss.backward()
                        optimizer.step()
                total += float(loss.detach())*len(x)
                count += len(x)
            row[name] = total/count
        history.append(row)
        if row["val"] < best:
            best = row["val"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    torch.save(best_state, Path(out)/f"refiner_d{dim}.pt")
    json_write(Path(out)/f"refine_history_d{dim}.json", history)
    with torch.no_grad():
        z = np.concatenate([model(torch.from_numpy(x).to(device))[0].cpu().numpy()
                            for x in np.array_split(h, max(1, math.ceil(len(h)/cfg.batch_size)))])
    print(f"NSV refine d={dim}: best validation normalized-latent MSE={best:.5f}", flush=True)
    return z


def run_training(root, dataset_id, condition, cache_root, cfg, method, device=None):
    """One immutable run; test states never enter optimization or hyperparameter selection."""
    cfg.validate()
    if method not in {"nsv", "infonce", "bt_temporal", "bt_same"}:
        raise ValueError(method)
    spec = CONDITIONS[condition]
    bundle, sim, manifest = load_dataset(root, dataset_id, spec["kind"])
    identity = {"dataset_id": dataset_id, "condition": condition, "condition_spec": spec,
                "method": method, "training": asdict(cfg), "data_sha": manifest["data_sha256"][spec["kind"]],
                "learning_source_sha": file_sha(__file__)}
    run_id = f"{method}_s{cfg.seed}_{digest(identity)}"
    out = Path(root)/"runs"/dataset_id/condition/run_id
    if (out/"complete.json").exists():
        complete = json.loads((out/"complete.json").read_text())
        if file_sha(out/"features.npz") != complete["features_sha256"]:
            raise ValueError("Completed feature artifact was modified.")
        print("Reusing completed run:", out)
        return out
    if out.exists():
        raise FileExistsError(f"Incomplete run at {out}. Inspect it or change run_tag; no silent overwrite.")
    seed_all(cfg.seed)
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    # Construct/validate windows before creating a run directory.
    train_index = index_windows(bundle, cfg.window, cfg.lag, cfg.train_stride)
    eval_index = index_windows(bundle, cfg.window, 2*sim.frames_per_period, cfg.eval_stride)
    frames = frame_cache(bundle, sim, condition, cache_root, dataset_id)
    mode = "predict" if method == "nsv" else ("same" if method == "bt_same" else "temporal")
    train_splits = bundle["splits"][train_index[:, 0]]
    train_ds = WindowDataset(frames, train_index[train_splits == 0], condition, sim.seed, cfg, mode, True)
    val_ds = WindowDataset(frames, train_index[train_splits == 1], condition, sim.seed, cfg, mode, True)
    if min(len(train_ds), len(val_ds)) < 2:
        raise ValueError("Insufficient windows per split.")
    out.mkdir(parents=True)
    runtime = {"python": platform.python_version(), "torch": str(torch.__version__),
               "numpy": np.__version__, "device": str(device), "cuda": torch.version.cuda}
    json_write(out/"run.json", {**identity, "run_id": run_id, "runtime": runtime,
                               "selection": "validation objective only; no test selection",
                               "temporal_window_overlap_fraction": max(0, cfg.window-cfg.lag)/cfg.window})
    model = Representation(cfg, predictive=(method == "nsv"))
    fit_representation(model, train_ds, val_ds, method, cfg, device, out)
    ds = WindowDataset(frames, eval_index, condition, sim.seed, cfg)
    h, z = encode(model, ds, device, cfg.batch_size, predictive=(method == "nsv"))
    splits = bundle["splits"][eval_index[:, 0]]
    groups = bundle["groups"][eval_index[:, 0]]
    features = {"index": eval_index, "h": h}
    diagnostic = estimate_id(h[splits == 0], groups[splits == 0], seed=cfg.seed)
    json_write(out/"id_train.json", diagnostic)
    if method == "nsv":
        for dim in cfg.refine_dims:
            features[f"z_d{dim}"] = refine(h, splits, dim, cfg, device, out)
    else:
        features["z"] = z
        if method == "infonce":
            features["z_unit"] = z/np.maximum(np.linalg.norm(z, axis=1, keepdims=True), 1e-12)
    np.savez_compressed(out/"features.npz", **features)
    json_write(out/"complete.json", {"run_id": run_id, "features_sha256": file_sha(out/"features.npz")})
    return out


## Cell 3 — Matched training configuration


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from dataclasses import replace
from l1_data import CONDITIONS, load_dataset, frame_cache, observation
from l1_learning import TrainConfig, Representation, run_training

DATASET_ID = ''  # paste from notebook 39; blank works only if exactly one dataset exists
CONDITIONS_TO_RUN = ['L1a']  # later: L1b_s002, L1b_s005, L1c_visible, L1c_hidden
SEEDS = [42]  # pilot first; then [42, 43, 44] with the same list in notebook 41
BASE = TrainConfig(window=5, lag=5, epochs=30, refine_epochs=60,
                   batch_size=128, view_noise_std=0.01, run_tag='pilot')
DATASET_ID = resolve_dataset_id(DATASET_ID)
if not torch.cuda.is_available():
    raise RuntimeError('Select Colab Runtime > Change runtime type > T4 GPU for training.')
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)
print('Dataset:', DATASET_ID, '| conditions:', CONDITIONS_TO_RUN)
for condition in CONDITIONS_TO_RUN:
    if condition not in CONDITIONS:
        raise ValueError(condition)


In [ ]:
METHODS = ['infonce', 'bt_temporal', 'bt_same']
PROJECTION_DIMS = [2]  # add 3/4/8 for a dimension sweep; include 3 for L1c


## Cell 4 — Train each objective from the same encoder initialization


In [ ]:
RUNS = []
for condition in CONDITIONS_TO_RUN:
    for seed in SEEDS:
        for dim in PROJECTION_DIMS:
            for method in METHODS:
                config = replace(BASE, seed=seed, projection_dim=dim)
                path = run_training(L1_ROOT, DATASET_ID, condition, CACHE_DIR, config, method)
                RUNS.append(path)
                print('Completed:', path)


## Cell 5 — Objective curves and train-only collapse diagnostics


In [ ]:
for path in RUNS:
    meta = json.loads((path / 'run.json').read_text())
    history = pd.read_json(path / 'training_history.json')
    history.plot(x='epoch', y=['train', 'val'], title=path.name)
    plt.ylabel('Objective value (not comparable across methods)'); plt.show()
    bundle, _, _ = load_dataset(L1_ROOT, DATASET_ID, CONDITIONS[meta['condition']]['kind'])
    with np.load(path / 'features.npz') as f:
        train = bundle['splits'][f['index'][:, 0]] == 0
        for name in ['h', 'z']:
            z = f[name][train]
            print(name, 'std min/median/max:', np.min(z.std(0)), np.median(z.std(0)), np.max(z.std(0)))
    display(pd.DataFrame(json.loads((path / 'id_train.json').read_text())['scales']))
print('Next: notebook 42. Training loss alone does not establish physical state recovery.')
